worked on by:

Building models

The previous part ended with a single notebook (per dataset) that would prepare your data for predicting. Next is building a couple of models and actually predicting something.

Which models will you need?

    A quick first model. This won't be a good one, but with this you can start working on the deployment (next step) while still tuning the model.
    Use PyCaret (or another automated ML comparison) on both datasets.
    Create and tune a model on both your datasets. Explain why you choose this particular model and perhaps train a second model to validate this choice.
    Create and tune a model on AWS.

Make sure to keep all the metrics on the models you made and compare these to show which model performed best.

# Predict

In [166]:
%pip install pandas matplotlib numpy
%pip install openpyxl
%pip install pycaret

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [167]:
import pickle
with open("cleaned_data.pkl", "rb") as file:
    df = pickle.load(file)


In [168]:
df.head()

,settlement_date,settlement_period,nd,tsd,england_wales_demand,embedded_wind_generation,embedded_wind_capacity,embedded_solar_generation,embedded_solar_capacity,non_bm_stor,pump_storage_pumping,ifa_flow,britned_flow,moyle_flow
0,2006-01-01,1,38596,39660.0,34982,0.0,0.0,0.0,0.0,0,295,1997,0.0,-169.0
1,2006-01-01,2,38829,39897.0,35312,0.0,0.0,0.0,0.0,0,299,1997,0.0,-169.0
2,2006-01-01,3,38456,39599.0,35018,0.0,0.0,0.0,0.0,0,374,1998,0.0,-169.0
3,2006-01-01,4,37401,38823.0,34054,0.0,0.0,0.0,0.0,0,653,1998,0.0,-169.0
4,2006-01-01,5,36586,37937.0,33297,0.0,0.0,0.0,0.0,0,582,1998,0.0,-169.0


In [169]:
import pandas as pd
df['year'] = pd.DatetimeIndex(df['settlement_date']).year
df['month'] = pd.DatetimeIndex(df['settlement_date']).month
df['day'] = pd.DatetimeIndex(df['settlement_date']).day
df = df.drop(columns=['settlement_date'])

In [170]:
df_monthly = df.drop(columns=["day"]).groupby(["year","month"], as_index=False).sum()


In [171]:
df_monthly.head()

,year,month,settlement_period,nd,tsd,england_wales_demand,embedded_wind_generation,embedded_wind_capacity,embedded_solar_generation,embedded_solar_capacity,non_bm_stor,pump_storage_pumping,ifa_flow,britned_flow,moyle_flow
0,2006,1,36456,66850752,68883004.0,60538297,0.0,0.0,0.0,0.0,3709,724022,1731921,0.0,-261166.0
1,2006,2,32928,60581983,62521698.0,54912731,0.0,0.0,0.0,0.0,3755,623125,802978,0.0,-125128.0
2,2006,3,36361,65731193,67562932.0,59480427,0.0,0.0,0.0,0.0,2406,694242,2118848,0.0,-184831.0
3,2006,4,35280,54662587,55996295.0,49382973,0.0,0.0,0.0,0.0,3,392020,2738382,0.0,-220748.0
4,2006,5,36456,53187407,54642331.0,48278122,0.0,0.0,0.0,0.0,1612,517304,2587698,0.0,-185414.0


In [173]:
df_monthly.tail()

,year,month,settlement_period,nd,tsd,england_wales_demand,embedded_wind_generation,embedded_wind_capacity,embedded_solar_generation,embedded_solar_capacity,non_bm_stor,pump_storage_pumping,ifa_flow,britned_flow,moyle_flow
233,2025,6,35280,31460339,35200409.0,29006861,2883536.0,9512640.0,5184445.0,29929765.0,0,186887,1447052,41775.0,-388454.0
234,2025,7,36456,33726450,37045701.0,30884743,1767780.0,9829728.0,4706975.0,31091242.0,0,139751,2291478,-64734.0,-261427.0
235,2025,8,36456,32450987,36019477.0,29741640,2353570.0,9829728.0,4328802.0,30349392.0,0,127016,2514884,71486.0,-413406.0
236,2025,9,35280,34026039,37663006.0,31296493,2934689.0,9512640.0,3301690.0,30229920.0,0,264064,1476418,-133266.0,-395658.0
237,2025,10,3528,3695627,4025846.0,3455810,448099.0,951264.0,136476.0,3022992.0,0,21601,74357,-37055.0,-12308.0


In [174]:
df_monthly = df_monthly.iloc[:-1]

In [175]:
df_monthly.tail()

,year,month,settlement_period,nd,tsd,england_wales_demand,embedded_wind_generation,embedded_wind_capacity,embedded_solar_generation,embedded_solar_capacity,non_bm_stor,pump_storage_pumping,ifa_flow,britned_flow,moyle_flow
232,2025,5,36456,32299365,35623800.0,29436302,2247016.0,9829728.0,5339143.0,30532262.0,0,101952,1972434,28874.0,-542754.0
233,2025,6,35280,31460339,35200409.0,29006861,2883536.0,9512640.0,5184445.0,29929765.0,0,186887,1447052,41775.0,-388454.0
234,2025,7,36456,33726450,37045701.0,30884743,1767780.0,9829728.0,4706975.0,31091242.0,0,139751,2291478,-64734.0,-261427.0
235,2025,8,36456,32450987,36019477.0,29741640,2353570.0,9829728.0,4328802.0,30349392.0,0,127016,2514884,71486.0,-413406.0
236,2025,9,35280,34026039,37663006.0,31296493,2934689.0,9512640.0,3301690.0,30229920.0,0,264064,1476418,-133266.0,-395658.0


In [176]:
df_monthly = df_monthly.drop('settlement_period', axis=1)

In [177]:
df_monthly.head()

,year,month,nd,tsd,england_wales_demand,embedded_wind_generation,embedded_wind_capacity,embedded_solar_generation,embedded_solar_capacity,non_bm_stor,pump_storage_pumping,ifa_flow,britned_flow,moyle_flow
0,2006,1,66850752,68883004.0,60538297,0.0,0.0,0.0,0.0,3709,724022,1731921,0.0,-261166.0
1,2006,2,60581983,62521698.0,54912731,0.0,0.0,0.0,0.0,3755,623125,802978,0.0,-125128.0
2,2006,3,65731193,67562932.0,59480427,0.0,0.0,0.0,0.0,2406,694242,2118848,0.0,-184831.0
3,2006,4,54662587,55996295.0,49382973,0.0,0.0,0.0,0.0,3,392020,2738382,0.0,-220748.0
4,2006,5,53187407,54642331.0,48278122,0.0,0.0,0.0,0.0,1612,517304,2587698,0.0,-185414.0


In [178]:
from pycaret.time_series import *
s = setup(
    data=df_monthly,
    target="england_wales_demand",
    fh=6,          
    session_id=123,
    fold=5
)

,Description,Value
0,session_id,123
1,Target,england_wales_demand
2,Approach,Univariate
3,Exogenous Variables,Present
4,Original data shape,"(237, 14)"
5,Transformed data shape,"(237, 14)"
6,Transformed train set shape,"(231, 14)"
7,Transformed test set shape,"(6, 14)"
8,Rows with missing values,0.0%
9,Fold Generator,ExpandingWindowSplitter


In [179]:
best = compare_models()

,Model,MASE,RMSSE,MAE,RMSE,MAPE,SMAPE,R2,TT (Sec)
auto_arima,Auto ARIMA,0.0634,0.0572,110400.0124,129202.4856,0.0031,0.0031,0.9953,38.1340
arima,ARIMA,0.0695,0.0657,121008.7519,148227.0584,0.0033,0.0033,0.9936,1.8400
huber_cds_dt,Huber w/ Cond. Deseasonalize & Detrending,0.5544,0.4997,962240.5298,1124556.6411,0.0265,0.0267,0.6600,0.1120
en_cds_dt,Elastic Net w/ Cond. Deseasonalize & Detrending,0.5986,0.5330,1039872.5089,1200459.3908,0.0291,0.0294,0.5196,0.3040
lightgbm_cds_dt,Light Gradient Boosting w/ Cond. Deseasonalize & Detrending,0.6082,0.5490,1056938.9366,1237087.7562,0.0297,0.0299,0.5289,0.2140
ridge_cds_dt,Ridge w/ Cond. Deseasonalize & Detrending,0.6215,0.5437,1080599.7759,1225688.8739,0.0300,0.0300,0.5519,0.2960
lr_cds_dt,Linear w/ Cond. Deseasonalize & Detrending,0.6222,0.5446,1081788.1718,1227819.6417,0.0300,0.0301,0.5516,1.1880
llar_cds_dt,Lasso Least Angular Regressor w/ Cond. Deseasonalize & Detrending,0.6222,0.5446,1081787.7547,1227818.5397,0.0300,0.0301,0.5516,0.1000
lasso_cds_dt,Lasso w/ Cond. Deseasonalize & Detrending,0.6227,0.5448,1082614.3430,1228363.6924,0.0300,0.0301,0.5512,0.2620
br_cds_dt,Bayesian Ridge w/ Cond. Deseasonalize & Detrending,0.6272,0.5541,1088405.6070,1246767.0797,0.0304,0.0309,0.4948,0.1020


In [180]:
with open('best_model.pkl', 'wb') as file:
    pickle.dump(best, file)

In [181]:
plot_model(best, plot = 'forecast', data_kwargs = {'fh' : 6})